# 🚗 Evaluación del Agente — Duckietown (Deep RL)

**Notebook autocontenido** para cargar `best_duckie_agent.zip` y evaluarlo, generando un vídeo de la conducción.

Cumple el **Contrato de Evaluación**:
- Observación `(1, 64, 64)` apilada en 4 frames → `(4, 64, 64)`.
- Incluye **inline** las definiciones exactas de `DuckieWrapper` y `CustomCNN` (no se importan de `train.py`), de modo que el modelo se cargue a la primera en un entorno limpio.

> El profesor cambiará únicamente el mapa a `Duckietown-loop_obstacles-v0`.

## 1. Instalación de dependencias (Google Colab, Python 3.11)
Ejecutar una sola vez. Reiniciar el entorno de ejecución si Colab lo solicita.

In [ ]:
# --- Solo en Google Colab: instalar dependencias del sistema y Python ---
# (Descomentar para ejecutar)
# !apt-get update -qq
# !apt-get install -y -qq xvfb freeglut3-dev libosmesa6-dev > /dev/null
# !pip install -q -r requirements.txt
# !pip install -q git+https://github.com/duckietown/gym-duckietown.git@daffy
print('Dependencias listas (ver requirements.txt).')

## 2. Imports y pantalla virtual

In [ ]:
import os
import numpy as np
import pyvirtualdisplay
import gym as old_gym
import gymnasium as gym
from gymnasium import spaces
import gym_duckietown
import cv2
import torch
import torch.nn as nn
from stable_baselines3 import PPO, SAC, DQN
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

# Pantalla virtual (obligatoria en Colab para el render OpenGL)
display = pyvirtualdisplay.Display(visible=False, size=(800, 600))
display.start()
print('OK')

## 3. Definiciones EXACTAS de clases (Wrapper + CNN)
Deben coincidir con las usadas en el entrenamiento para que `model.load()` funcione.

In [ ]:
IMG_SIZE = 64
N_STACK = 4
OBS_SHAPE = (1, IMG_SIZE, IMG_SIZE)


class DuckieWrapper(gym.Env):
    """Adapta gym-duckietown a Gymnasium: recorta cielo, gris, 64x64 -> (1,64,64)."""
    metadata = {"render_modes": ["rgb_array"]}

    def __init__(self, env_name="Duckietown-loop_empty-v0", seed=None):
        super().__init__()
        self.env_name = env_name
        self.env = old_gym.make(env_name)
        if seed is not None:
            try:
                self.env.seed(seed)
            except Exception:
                pass
        self.action_space = spaces.Box(
            low=np.array([-1.0, -1.0], dtype=np.float32),
            high=np.array([1.0, 1.0], dtype=np.float32), dtype=np.float32)
        self.observation_space = spaces.Box(low=0, high=255, shape=OBS_SHAPE, dtype=np.uint8)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        obs = self.env.reset()
        if isinstance(obs, tuple):
            obs = obs[0]
        return self._process_obs(obs), {}

    def step(self, action):
        action = np.asarray(action, dtype=np.float32).reshape(-1)
        obs, reward, done, info = self.env.step(action)
        return self._process_obs(obs), float(reward), bool(done), False, info

    def _process_obs(self, obs):
        obs = obs[obs.shape[0] // 2:, :, :]
        gray = cv2.cvtColor(obs, cv2.COLOR_RGB2GRAY)
        resized = cv2.resize(gray, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        return np.expand_dims(resized, axis=0).astype(np.uint8)

    def render(self):
        return self.env.render(mode="rgb_array")

    def close(self):
        self.env.close()


class CustomCNN(BaseFeaturesExtractor):
    """CNN tipo Nature adaptada a (4,64,64) con normalización de píxeles."""
    def __init__(self, observation_space, features_dim=256):
        super().__init__(observation_space, features_dim)
        n_input_channels = observation_space.shape[0]
        self.cnn = nn.Sequential(
            nn.Conv2d(n_input_channels, 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1), nn.ReLU(),
            nn.Flatten())
        with torch.no_grad():
            sample = torch.as_tensor(observation_space.sample()[None]).float()
            n_flatten = self.cnn(sample).shape[1]
        self.linear = nn.Sequential(nn.Linear(n_flatten, features_dim), nn.ReLU())

    def forward(self, observations):
        observations = observations.float() / 255.0
        return self.linear(self.cnn(observations))


print('Clases definidas: DuckieWrapper, CustomCNN')

## 4. Carga del agente y evaluación con generación de vídeo
El profesor cambiará `MAP_NAME` a `Duckietown-loop_obstacles-v0`.

In [ ]:
import imageio
from IPython.display import Video

MAP_NAME = "Duckietown-small_loop-v0"   # Evaluación final: 'Duckietown-loop_obstacles-v0'
MODEL_PATH = "../models/best_duckie_agent"  # Colab: sube best_duckie_agent.zip al mismo dir y usa "best_duckie_agent"
ALGO = SAC   # algoritmo con el que se guardó el mejor agente (SAC o PPO)


def make_test_env():
    return DuckieWrapper(MAP_NAME)


test_env = DummyVecEnv([make_test_env])
test_env = VecFrameStack(test_env, n_stack=N_STACK)

model = ALGO.load(MODEL_PATH)

print("Evaluando agente...")
obs = test_env.reset()
frames = []
total_reward = 0.0
for i in range(1000):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, info = test_env.step(action)
    total_reward += float(reward[0])
    frames.append(test_env.envs[0].env.render(mode="rgb_array"))
    if done[0]:
        print(f"Episodio finalizado en el paso {i}")
        break

test_env.close()
print(f"Recompensa acumulada: {total_reward:.2f} | frames: {len(frames)}")

video_path = "duckie_eval_video.mp4"
imageio.mimsave(video_path, frames, fps=30)
Video(video_path, embed=True)